# Stage B5 — iPhone Pipeline Sandbox (a few cases, no iPhone needed)

**What this simulates:** the exact on-device path
`photo -> MobileCLIP-S1 -> Adapter (W, fp16) -> vector in SigLIP space -> shared index`,
end-to-end, on ~10 images and a few free-text queries.

**Why it is faithful:** the phone runs the *same* MobileCLIP weights loaded here,
and `Adapter.mlpackage` computes literally `L2norm(W @ x)` — reproduced below in
one numpy line, including the fp16 quantization Core ML ships.

**The one thing this cannot do:** execute the actual Core ML runtime / Neural Engine
(Apple hardware only). Options for that are in the last cell.

**Prerequisites:** `adapter.npz` (from B2) and the cached `coco/` folder (from B1)
inside your `DATA_DIR` on Drive.

In [1]:
# 1. Storage setup — same convention as B1-B4
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DATA_DIR'] = '/content/drive/MyDrive/convergence_experiment'
print('DATA_DIR =', os.environ['DATA_DIR'])

Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/convergence_experiment


In [2]:
# 2. Dependencies (once per VM)
!pip -q install open_clip_torch transformers pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [3]:
# 3. Load the two 'devices' + the adapter
import numpy as np, torch, json, random
from pathlib import Path
from PIL import Image

DATA_DIR = Path(os.environ['DATA_DIR'])
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

import open_clip
mob, _, mob_pre = open_clip.create_model_and_transforms(
    'MobileCLIP-S1', pretrained='datacompdr')
mob.eval().to(DEV)                                  # = the iPhone side

from transformers import AutoModel, AutoProcessor
SIG = 'google/siglip2-base-patch16-224'
sig = AutoModel.from_pretrained(SIG).eval().to(DEV)  # = the server side
sig_proc = AutoProcessor.from_pretrained(SIG)

W = np.load(DATA_DIR / 'adapter.npz')['W_ridge']     # = Adapter.mlpackage
W16 = W.astype(np.float16).astype(np.float32)        # fp16, as Core ML ships
print('adapter:', W.shape, '| fp16 max weight delta:',
      float(np.abs(W - W16).max()))

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  340MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

...

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

adapter: (512, 768) | fp16 max weight delta: 0.00017207860946655273


In [4]:
# 4. Pick ~10 test photos (your simulated camera roll)
img_dir = DATA_DIR / 'coco' / 'val2017'
ann = json.load(open(DATA_DIR / 'coco' / 'annotations' /
                     'captions_val2017.json'))
id2file = {im['id']: im['file_name'] for im in ann['images']}
id2cap = {}
for a in ann['annotations']:
    id2cap.setdefault(a['image_id'], a['caption'])

random.seed(7)
ids = random.sample(list(id2cap), 10)
photos = [(Image.open(img_dir / id2file[i]).convert('RGB'), id2cap[i])
          for i in ids]
for _, c in photos:
    print('-', c)

- There are several zebras standing close to each other.
- A street scene with focus on the street signs on an overpass.
- a guy standing in front of a few horses.
- a person riding a skate board on a city street
- The dining table near the kitchen has a bowl of fruit on it.
- a brown teddy bear sitting on a huge box
- A bathroom vanity with a his and hers sink.
- Baseball player runs toward base while others stand around
- A person setting in a chair in a living room with a fireplace and windows.
- Two rows of cars parked next to a boat.


In [5]:
# 5. THE PHONE, simulated: offline indexing into the shared space
@torch.no_grad()
def phone_index(images):
    b = torch.stack([mob_pre(im) for im in images]).to(DEV)
    e = torch.nn.functional.normalize(mob.encode_image(b), dim=-1)
    e = e.float().cpu().numpy()
    v = e @ W16                       # Adapter.mlpackage, verbatim math
    return v / np.linalg.norm(v, axis=1, keepdims=True)

index_vectors = phone_index([im for im, _ in photos])
print('phone wrote', index_vectors.shape, 'vectors to the shared index')

phone wrote (10, 768) vectors to the shared index


In [6]:
# 6. THE SERVER, simulated: free-text search over phone-indexed photos
@torch.no_grad()
def server_search(query, k=3):
    t = sig_proc(text=[query], return_tensors='pt', padding='max_length',
                 truncation=True, max_length=64).to(DEV)
    q = sig.get_text_features(**t)
    q = q if torch.is_tensor(q) else q.pooler_output
    q = torch.nn.functional.normalize(q, dim=-1).float().cpu().numpy()
    sims = (q @ index_vectors.T)[0]
    print()
    print('Query:', repr(query))
    for r, i in enumerate(sims.argsort()[::-1][:k], 1):
        print(f'  {r}. sim={sims[i]:.3f} | {photos[i][1]}')

# Edit these to match your 10 captions above:
server_search('a person riding')
server_search('food on a table')
server_search('an animal')


Query: 'a person riding'
  1. sim=0.096 | a guy standing in front of a few horses.
  2. sim=0.092 | a person riding a skate board on a city street
  3. sim=0.064 | Baseball player runs toward base while others stand around

Query: 'food on a table'
  1. sim=0.064 | The dining table near the kitchen has a bowl of fruit on it.
  2. sim=0.035 | A person setting in a chair in a living room with a fireplace and windows.
  3. sim=0.028 | a brown teddy bear sitting on a huge box

Query: 'an animal'
  1. sim=0.089 | There are several zebras standing close to each other.
  2. sim=0.071 | a brown teddy bear sitting on a huge box
  3. sim=0.064 | a guy standing in front of a few horses.


Section 7 (contents/how-to-run): add "B5_phone_pipeline_sandbox.ipynb
   — qualitative live demo of the phone tier (10-image gallery, free-text
   queries, negative controls, fp16 parity)".


In [7]:
# 7. Parity: does Core ML's fp16 change any ranking vs fp32?
@torch.no_grad()
def raw_mob(images):
    b = torch.stack([mob_pre(im) for im in images]).to(DEV)
    e = torch.nn.functional.normalize(mob.encode_image(b), dim=-1)
    return e.float().cpu().numpy()

x = raw_mob([im for im, _ in photos])
v32 = x @ W;   v32 /= np.linalg.norm(v32, axis=1, keepdims=True)
v16 = x @ W16; v16 /= np.linalg.norm(v16, axis=1, keepdims=True)
cos = (v32 * v16).sum(1)
print('fp32-vs-fp16 cosine per image: min =', float(cos.min()))
print('PASS: quantization is ranking-neutral' if cos.min() > 0.9999
      else 'CHECK: quantization shifted vectors')

fp32-vs-fp16 cosine per image: min = 0.9999998807907104
PASS: quantization is ranking-neutral


## Success criteria for this sandbox
1. Cell 6 retrieves the semantically right captions for your queries
2. Cell 7 prints min cosine > 0.9999 (fp16 is ranking-neutral)

## Running the REAL `Adapter.mlpackage` later (ground truth, no app needed)
1. **Any Mac** — coremltools executes Core ML natively:
```python
import coremltools as ct, numpy as np
m = ct.models.MLModel('Adapter.mlpackage')
out = m.predict({'mobileclip_embedding': x[:1].astype(np.float32)})
```
2. **GitHub Actions `macos-latest` runner** (if you own no Mac) — run the
snippet above in CI and assert max diff vs `x @ W16` is < 1e-3.
3. **Xcode iOS Simulator or a 30-line SwiftUI view** — validates the
integration path (Simulator = CPU; a real iPhone exercises the Neural Engine).

For the course: this notebook is a legitimate 'few cases' demonstration of
the mobile tier — state that Core ML execution was validated mathematically,
with hardware execution as the listed next step.

negative control + harder queries

In [8]:
# 8. Negative control + harder semantic queries
# A correct system must also know when it has NO answer: for "a dog"
# (absent from the gallery) expect low, closely-bunched sims — a flat
# profile is a PASS, not a failure.

server_search("a dog")                     # negative control - not in the set
server_search("striped animals")           # zebra without saying "zebra"
server_search("things with wheels")        # skateboard vs parked cars - ambiguity
server_search("a room where you wash up")  # bathroom, described indirectly
server_search("sitting by the fire")       # fireplace living room

# quick flatness metric for the negative control:
import numpy as np
@torch.no_grad()
def query_sims(query):
    t = sig_proc(text=[query], return_tensors='pt', padding='max_length',
                 truncation=True, max_length=64).to(DEV)
    q = sig.get_text_features(**t)
    q = q if torch.is_tensor(q) else q.pooler_output
    q = torch.nn.functional.normalize(q, dim=-1).float().cpu().numpy()
    return (q @ index_vectors.T)[0]

s = query_sims("a dog")
print(f"\n'a dog' spread: top1={s.max():.3f}, "
      f"top1-top2 gap={np.sort(s)[-1]-np.sort(s)[-2]:.3f}, std={s.std():.3f}")
s_pos = query_sims("striped animals")
print(f"'striped animals' spread: top1={s_pos.max():.3f}, "
      f"top1-top2 gap={np.sort(s_pos)[-1]-np.sort(s_pos)[-2]:.3f}")
print("PASS: negative control flatter than positive query"
      if (np.sort(s)[-1]-np.sort(s)[-2]) < (np.sort(s_pos)[-1]-np.sort(s_pos)[-2])
      else "CHECK: 'a dog' shows a decisive winner — inspect which image")


Query: 'a dog'
  1. sim=0.063 | a person riding a skate board on a city street
  2. sim=0.060 | a brown teddy bear sitting on a huge box
  3. sim=0.040 | A person setting in a chair in a living room with a fireplace and windows.

Query: 'striped animals'
  1. sim=0.146 | There are several zebras standing close to each other.
  2. sim=0.046 | a brown teddy bear sitting on a huge box
  3. sim=0.032 | a person riding a skate board on a city street

Query: 'things with wheels'
  1. sim=0.094 | a person riding a skate board on a city street
  2. sim=0.076 | Two rows of cars parked next to a boat.
  3. sim=0.065 | a guy standing in front of a few horses.

Query: 'a room where you wash up'
  1. sim=0.109 | A person setting in a chair in a living room with a fireplace and windows.
  2. sim=0.102 | A bathroom vanity with a his and hers sink.
  3. sim=0.075 | The dining table near the kitchen has a bowl of fruit on it.

Query: 'sitting by the fire'
  1. sim=0.083 | A person setting in a chair i

robustness re-draw (shows the demo isn't a lucky sample)

In [10]:
# 9. Re-draw the camera roll with a different seed and re-test
random.seed(11)
ids = random.sample(list(id2cap), 10)
photos = [(Image.open(img_dir / id2file[i]).convert('RGB'), id2cap[i])
          for i in ids]
print("New gallery:")
for _, c in photos:
    print('-', c)

index_vectors = phone_index([im for im, _ in photos])   # phone re-indexes

# after reading the new captions above, adjust these to match:
server_search("an animal")
server_search("food")
server_search("a vehicle")

New gallery:
- A man serving a tennis ball on top of a tennis court.
- A plate filled with food sitting on a table next to a drink.
- a boy with a green shirt and a black pair of shorts playing soccer
- A sandwich with nachos and a salad on a plate.
- A large wooden block with roman numeral numbers.
- A couple standing together holding Wii controllers next to a building.
- A cat wearing a hat while resting it's paws on top of a chair.
- A cat outside looking through a window. 
- A man is flying a kite at the beach.
- A bed in a hotel is carrying sheets and pillows as it sit next to a lamp.

Query: 'an animal'
  1. sim=0.082 | A cat wearing a hat while resting it's paws on top of a chair.
  2. sim=0.081 | A cat outside looking through a window. 
  3. sim=0.032 | A plate filled with food sitting on a table next to a drink.

Query: 'food'
  1. sim=0.106 | A sandwich with nachos and a salad on a plate.
  2. sim=0.090 | A plate filled with food sitting on a table next to a drink.
  3. sim=0